# 第8回　前処理と Pipeline
***
> **前提**: 第5回（タイタニック）で学んだ前処理を発展させます。第5回は `dropna` で欠損を除去しましたが、本回は**補完**と **Pipeline** で自動化します。

## 目次
1. データの読み込み
2. 欠損値の確認と補完
3. ColumnTransformer と Pipeline
4. モデル評価

---

## この回で学ぶこと

### なぜ前処理が重要なのか

現実のデータは必ず「汚れている」。欠損値・外れ値・スケールの違いを放置したままモデルを学習させると，精度が著しく落ちたり，最悪の場合モデルが全く機能しない。前処理はデータサイエンスの作業時間の **60〜80%** を占めると言われており，卒業研究でも避けられないスキルだ。

### 欠損値補完（Imputation）

欠損値の扱いには大きく3つの方針がある：

| 方針 | 方法 | いつ使うか |
|---|---|---|
| 削除 | `dropna()` | 欠損が全体の5%未満，かつランダムに欠損している場合 |
| 統計値で補完 | `SimpleImputer` | 欠損が多く削除できない場合（本回） |
| モデルで予測補完 | `IterativeImputer` など | 欠損パターンに意味がある場合（発展） |

`SimpleImputer` の `strategy` の違い：
- `"mean"` : 外れ値の影響を受けやすい。正規分布に近いデータに向く
- `"median"` : 外れ値に強い。**Age のような右に歪んだデータに推奨**
- `"most_frequent"` : カテゴリ変数や欠損率が低い数値列に向く

### スケーリング（正規化・標準化）

多くの機械学習アルゴリズム（SVM, KNN, ニューラルネット, 正則化回帰）は，特徴量のスケールに敏感だ。例えば `Fare`（0〜500程度）と `Pclass`（1〜3）をそのまま使うと，Fare の影響が過大になる。

| 手法 | 変換式 | 特徴 |
|---|---|---|
| `StandardScaler` | (x - μ) / σ | 平均0，標準偏差1。外れ値があっても比較的安定 |
| `MinMaxScaler` | (x - min) / (max - min) | 0〜1 に圧縮。外れ値に弱い |

決定木・ランダムフォレストは距離を使わないのでスケーリング不要だが，今回は Pipeline の練習として適用する。

### Pipeline の本当の価値：データリーク防止

Pipeline を使わずに手動で `fit_transform` すると，テストデータの情報が訓練データの統計量に混入する「**データリーク（Data Leakage）**」が起きる危険がある。

```
【NG例：データリーク】
scaler.fit_transform(X_all)  # テストデータの情報も含めて統計量を計算 → NG
X_train, X_test = split(X_all_scaled)

【OK例：Pipeline】
pipeline.fit(X_train, y_train)   # 訓練データだけで統計量を計算
pipeline.predict(X_test)          # 訓練データの統計量でテストデータを変換 → OK
```

卒業研究で「時系列データの未来情報が混入してモデルが異常に高精度になった」という失敗はよく起こる。Pipeline はこれを防ぐ設計思想だ。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

import pandas as pd

TITANIC_URL = "https://raw.githubusercontent.com/ShotaYmzk/AI-kadai/main/data/titanic/titanic.csv"

def load_titanic() -> pd.DataFrame:
    return pd.read_csv(TITANIC_URL)


: 

## 問題1　データの読み込みと欠損確認
***

### 背景

分析を始める前に必ず「データの全体像を把握する」ことが重要だ。列の型・サンプル数・欠損数を確認しないまま進めると，後で予期しないエラーや誤った分析に繋がる。これは研究でも実務でも最初に必ずやるべきステップだ。

各列の意味：
- `Survived` : 生存フラグ（0=死亡, 1=生存）→ **予測したい目的変数**
- `Pclass` : 客室クラス（1=一等, 2=二等, 3=三等）→ 社会的地位の代理変数
- `Sex` : 性別（male/female）→ カテゴリ変数なので後でエンコードが必要
- `Age` : 年齢 → **欠損が多い列（約20%）**
- `SibSp` / `Parch` : 同乗する兄弟姉妹数 / 親子数
- `Fare` : 運賃 → 右に裾が長い分布（外れ値あり）

### 課題

タイタニック号の乗客データを URL から読み込み，`Survived`・`Pclass`・`Sex`・`Age`・`SibSp`・`Parch`・`Fare` の列のみを抽出した DataFrame `df_titanic` を作成してください。

データの**形式**（`.info()` または `.shape`）と**各列の欠損数**（`.isnull().sum()`）を確認してください。

> **確認ポイント**: 欠損が特定の列に集中していないか？欠損は「ランダムに発生している」のか「特定のグループに偏っている」のかを考えることが，補完方法の選択に直結する。

In [ ]:
# データの読み込み
# ここにあなたのコードを書いてください


# 形式と欠損の確認
# ここにあなたのコードを書いてください


## 問題2　欠損値補完
***

### なぜ中央値を使うのか

`Age` 列の分布は右に歪んでいる（若者が多く，高齢者が少ない）。このような分布では：
- **平均値（mean）** は外れ値（100歳超など）に引っ張られて実態より高くなる
- **中央値（median）** は順位に基づくため，外れ値の影響を受けない

> **研究での判断基準**: ヒストグラムや箱ひげ図で分布の歪みを確認してから補完方法を選ぼう。`df["Age"].hist()` で確認できる。

### `fit` と `transform` の分離が重要な理由

```python
imputer.fit(X_train)      # 訓練データから中央値を「学習」
imputer.transform(X_test) # 学習した中央値でテストデータを変換
```

`fit_transform(X_all)` を全データに使うと，テストデータの値も中央値の計算に含まれる（データリーク）。今回は練習のため全体に適用するが，実際の分析では必ず訓練データだけで `fit` すること。

### 課題

`Age` 列の欠損値を**中央値**で補完してください。`SimpleImputer` を用い，補完前後の欠損数を出力してください。

#### Hints
- 補完には `SimpleImputer` クラスを使う。`strategy` 引数で補完方法を選ぶ
- `fit_transform` の返り値は **numpy 配列**（DataFrame ではない）。列への代入時はデータの形状に気をつける
- 補完後は `isnull().sum()` で欠損がゼロになったか確認する習慣をつけよう


In [ ]:
# Age列の欠損補完
# ここにあなたのコードを書いてください


## 問題3　ColumnTransformer と Pipeline の構築
***

### ColumnTransformer とは

現実のデータには数値列とカテゴリ列が混在している。それぞれに異なる前処理を適用するのが `ColumnTransformer` だ。

```
【前処理の分岐イメージ】

数値列 (Pclass, Age, SibSp, Parch, Fare)
    → StandardScaler: スケールを揃える（平均0，標準偏差1）

カテゴリ列 (Sex: "male"/"female")
    → OneHotEncoder: 数値に変換（male=1,0 / female=0,1）
    ※ drop_first=True で多重共線性を避けるため片方を削除することも多い
```

### OneHotEncoding が必要な理由

機械学習モデルは文字列（"male", "female"）を直接扱えない。`LabelEncoder` で 0/1 に変換する方法もあるが，「male=0, female=1」という**数値の大小関係**が意味を持ってしまうため，名義尺度には OneHotEncoding が適切だ。

### criterion="entropy" の意味

決定木には分岐基準として `gini`（デフォルト）と `entropy` がある。どちらも「不純度」を測る指標だが：
- `gini` : 計算が速い
- `entropy` : 情報理論に基づく，不均衡クラスにやや強い

実用上は大差ないが，`max_depth=5` で木の深さを制限することが**過学習防止**に効いている（第13回で詳しく学ぶ）。

### 課題

数値列（`Pclass`, `Age`, `SibSp`, `Parch`, `Fare`）には `StandardScaler`，カテゴリ列（`Sex`）には `OneHotEncoder` を適用する `ColumnTransformer` を作成してください。

その `ColumnTransformer` と `DecisionTreeClassifier(criterion="entropy", max_depth=5)` を `Pipeline` で結合し，訓練データ（80%）で学習してください。

#### Hints
- 目的変数 `y` と説明変数 `X` を `drop` で分離してから `train_test_split` に渡す
- `ColumnTransformer` は `(名前, 変換器オブジェクト, 列名のリスト)` のタプルのリストで構成する
- `Pipeline` は `[(ステップ名, 変換器/モデル), ...]` のリストを受け取る。`ColumnTransformer` を第一ステップ、`DecisionTreeClassifier` を第二ステップにする
- `pipeline.fit()` を呼ぶだけで前処理とモデル学習がまとめて実行される

In [ ]:
# ColumnTransformer の作成
# ここにあなたのコードを書いてください


# Pipeline の作成と学習
# ここにあなたのコードを書いてください


## 問題4　モデル評価
***

### 正解率（Accuracy）だけで判断してはいけない

正解率はわかりやすい指標だが，**クラスが不均衡な場合に誤解を招く**。例えばタイタニックでは生存者（38%）より死亡者（62%）が多い。「全員死亡」と予測するだけで正解率62%になる。

そのため実際の研究では正解率に加えて以下も確認する：
- **適合率（Precision）**: 「生存と予測した中で本当に生存した割合」
- **再現率（Recall）**: 「実際の生存者の中でモデルが生存と予測した割合」
- **F1スコア**: Precision と Recall の調和平均

（これらは第12回で詳しく扱う。今回は正解率のみ求める。）

### Pipeline の predict の動作

`pipeline.predict(X_test)` を呼び出すと，内部で以下が**自動的に**行われる：
1. `ColumnTransformer` でテストデータを変換（StandardScaler は訓練データの μ,σ を使用）
2. 変換後のデータを `DecisionTreeClassifier` に入力して予測

これが Pipeline の強力さだ。手動で変換すると手順を間違えやすいが，Pipeline は常に正しい順序で処理する。

### 課題

問題3で学習した Pipeline を用いて，テストデータ（20%）の**正解率**を `accuracy_score` で求めて出力してください。

得られた正解率を見て，「全員死亡と予測した場合の正解率（約62%）」と比較して，モデルが意味のある予測をしているかどうかを確認してみよう。

#### Hints
- Pipeline は `predict` メソッドを持つ。`X_test` を渡すだけで前処理→予測が自動で行われる
- `accuracy_score` の引数は `(正解ラベル, 予測ラベル)` の順
- **発展**: `classification_report` を使うと Precision/Recall/F1 もまとめて確認できる

In [ ]:
# テストデータでの正解率
# ここにあなたのコードを書いてください
